In [77]:
#import any necessary libraries
import base64
import json
import random
import requests
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from IPython.display import Audio, HTML, Image, display
import ipywidgets as widgets
import pandas as pd

In [78]:
# read animal dataset
animals_df = pd.read_csv('data/Animal Characteristics Data.csv')

# read scenario file
from pathlib import Path
scenarios = json.loads(Path('data/scenarios.json').read_text())

In [89]:
# Load credentials from your service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
# Fetch an access token (this is what gcloud auth print-access-token returns)
creds.refresh(Request())
token = creds.token

with open('keys/service.json', 'r') as f:
    keys = json.load(f)


def get_audio(animal, curr_event):
    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"Create a song about how the animal {animal["animal"].iloc[0]} died during {curr_event['name']}, when {curr_event['description']}" #ADD THE PROMPT HERE!!!
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    return data["outputs"][2]["data"]

In [42]:
#Select an animal (the game is started in here!!!)
def select_animal(change): 
    global animal
    name = text_input.value.strip().capitalize()
    animal = animals_df.loc[animals_df.animal == name]

    with output:
        output.clear_output(wait=True)
        if animal.empty:
            print("Animal not found. Try again!")
        else:
            print("You selected:", animal["animal"].iloc[0])
            #update to update the global animal and change the output?
            start_game(animal)

In [80]:
def display_stats():
    print(f"Animal: {animal["animal"].iloc[0]}")
    print(f"Health: {max(0, animal_hp)}")
    print(f"Energy: {max(0, animal_nrg)}")
    print(f"Score: {player_score}")
    print("")

In [81]:
def start_game(animal):
    global animal_hp, animal_nrg, player_score

    #reset the stats
    animal_hp, animal_nrg = set_stats(animal)
    player_score = 0

    # map the starting stats to the display bars
    hp_bar.max = animal_hp if animal_hp > 100 else 100
    nrg_bar.max = animal_nrg if animal_nrg > 100 else 100


    #init the scenarios 
    get_event()
    show_event()

In [100]:
# select a random scenario from json file
def get_event():
    global curr_event, choices
    curr_event = scenarios[random.choice(list(scenarios.keys()))]
    choices = curr_event["choices"]

#update player stats accordingly when a choice is made
def choice_selected(button, choice):
    global animal_hp, animal_nrg, player_score 
    animal_hp += (choice["health"] * 3)
    animal_nrg += (choice["energy"]) #modified to make it more realistic
    player_score += choice["score"]

    if animal_hp <= 0 or animal_nrg <= 0:
        with output: 
            output.clear_output(wait=True)
            display_stats()
            print("You died!")
            print(f'{curr_event["name"]}...')
            
            #display picture of animal but change the style so you know somehting happened
            display(HTML(f'<img src="{animal["image_url"].iloc[0]}" width="200" height="200" style="filter: sepia(100%) saturate(300%) hue-rotate(-50deg);" />'))

            print("A song in memoriam will appear here:")
            
            #play song!
            #print(curr_event['name'], curr_event)
            song = get_audio(animal, curr_event)
            mp3_bytes = base64.b64decode(song)
            display(Audio(mp3_bytes))

            ##(remove the please wait print statement here!!)
            
            print("Enter a new animal to play again...")
    else:  
        get_event()
        show_event()


#display the choice buttons and scenatio image
def show_event():
    # print the initial pre-round text layout
    with output: 

        #clear the prev output 
        output.clear_output(wait=True)

        update_stats_display()

        #add strings to widget (as HTML for a pretty layout)
        event_title = widgets.HTML(f"<h3>{curr_event['name']}</h3>")
        event_desc = widgets.HTML(f"<p>{curr_event['description']}</p>")


        # make buttons, if a button is clicked, apply the impact of the choice made
        buttons = []
        for choice in choices.values():
            button = widgets.Button(
                description=choice["description"], 
                layout=widgets.Layout(width='95%', margin='2px')
            )
            button.on_click(lambda b, c=choice: choice_selected(b, c))
            buttons.append(button)


        #add picture of animal & scenario to widgit
        animal_img = widgets.HTML(f'<img src="{animal["image_url"].iloc[0]}" width="180" height="180" style="border-radius: 8px; margin-right: 15px;" />')
        scenario_img = widgets.HTML(f'<img src="{curr_event["scenario_image"]}" width="180" height="180" style="border-radius: 8px; margin-right: 15px;" />')

        #drawing out the layout
        # cols
        # add stats to widget, stacked verically
        stats_disp = widgets.VBox([widgets.Label(value=f"Animal: {animal['animal'].iloc[0]}"), hp_bar, nrg_bar, score_label]) 
        # add scenario display, stacked vertically
        gameplay_disp = widgets.VBox([event_title, event_desc, widgets.VBox(buttons)], layout=widgets.Layout(width='60%'))
        
        # Rows
        # add animal image and stats
        top_row = widgets.HBox([animal_img, stats_disp], layout=widgets.Layout(align_items='center', margin='0 0 15px 0'))
        # add scenario image and buttons
        bottom_row = widgets.HBox([scenario_img, gameplay_disp], layout=widgets.Layout(align_items='flex-start'))
        
        # put everything together
        game_board = widgets.VBox([top_row, widgets.HTML("<hr style='border: 0.5px solid #ccc;'/>"), bottom_row])
        
        display(game_board)
        


In [83]:
starting_stat = 30

#find the percentile of a stat in a datast
def remap_num(animal, col):
    return animals_df[col].rank(pct=True).loc[animal.index[0]] * 100
    
# calculate player stats based on animal selection
def set_stats(animal):
    global animal_hp, animal_nrg, player_score
    
    # variable that cuts the percentile to prevent being too OP
    factor = 4
    
    # health
    animal_hp = round(
        starting_stat
        + (remap_num(animal, "weight_kg") 
        + remap_num(animal, "lifespan_years")
          ) / factor
    )
    
    animal_nrg = round(
        starting_stat 
        + (remap_num(animal, "speed_kmh")
        + (4 * animal["can_swim"].iloc[0])
        + (2 * animal["can_fly"].iloc[0])
        + (2 *animal["has_wings"].iloc[0])
          )/ factor
    )
    return animal_hp, animal_nrg

#create the stat display components
hp_bar = widgets.IntProgress(value= starting_stat, min=0, max=100, description= f"Health ({animal_hp}): ", bar_style='danger')
nrg_bar = widgets.IntProgress(value=starting_stat, min=0, max=100, description=f"Energy ({animal_nrg}): ", bar_style='warning')
score_label = widgets.Label(value="Score: 0")

def update_stats_display():
    hp_bar.value = max(0, animal_hp)
    nrg_bar.value = max(0, animal_nrg)
    hp_bar.description = f"Health ({hp_bar.value}): "
    nrg_bar.description = f"Energy ({nrg_bar.value}): "
    
    # Update the text on our score label
    score_label.value = f"Score: {player_score}"

In [84]:
#show image from url
def display_url(url):
    display(HTMLImage(url=url, width=200, height=200))

In [102]:
#THIS CELL RUNS THE GAME!!!!!!!!!
output = widgets.Output()

text_input = widgets.Text(placeholder = 'Enter an animal')
text_input.observe(select_animal, names="value")

display(text_input)
display(output)

Text(value='', placeholder='Enter an animal')

Output()

In [ ]:
#The cell above activates the game!














## Thanks for playing!!